# ICOS zarr store — variable explorer

Demonstrates reading the zarr v2 store produced by `fluxnet2zarr.py`
and plotting the same key variables shown in `explore_restructured.ipynb`.

| Group path | Time resolution | Key content |
|---|---|---|
| `SE-Svb/` (root) | Half-hourly | Met forcing, NEE, GPP, RECO, LE, H, soil, METEOSENS profiles |
| `SE-Svb/fluxnet_dd` | Daily | Same variables aggregated to daily |
| `SE-Svb/fluxnet_mm` | Monthly | Monthly means |
| `SE-Svb/fluxnet_ww` | Weekly | Weekly means |
| `SE-Svb/fluxnet_yy` | Annual | Annual sums / means |

In [ ]:
import pathlib
import json
import numpy as np
import zarr
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

STORE = pathlib.Path("icos-fluxnet.zarr")
SITE  = "SE-Svb"

# consolidated=False: read individual .zattrs files directly, bypassing the
# potentially stale .zmetadata cache (avoids KeyError on custom attrs like _provenance)
z = zarr.open_group(str(STORE), mode="r", consolidated=False)

print("Stations  :", sorted(z.group_keys()))
print("Sub-groups:", sorted(z[SITE].group_keys()))
print("Root arrays:", len(list(z[SITE].array_keys())))

prov = json.loads(z[SITE].attrs["_provenance"])
print(f"\n── Provenance for {SITE} ──")
print(f"  Created      : {prov['created']}")
print(f"  Last updated : {prov['last_updated']}")
print(f"  Archive      : {prov['archive']}")
print(f"  Source DOI   : {prov['source_doi']}")
print(f"  Citation     : {prov['citation'][:120]}…")
print(f"  History      :")
for h in prov["history"]:
    print(f"    {h['timestamp']}  {h['action']:8s}  {h['archive']}")

## 0 · Dataset structure

Open each group with xarray and inspect dimensions, coordinates, and variable counts.

In [ ]:
# Open all groups with xarray (consolidated=False: always read from .zattrs, not .zmetadata cache)
ds_hh = xr.open_zarr(str(STORE), group=f"{SITE}",            consolidated=False)
ds_dd = xr.open_zarr(str(STORE), group=f"{SITE}/fluxnet_dd", consolidated=False)
ds_mm = xr.open_zarr(str(STORE), group=f"{SITE}/fluxnet_mm", consolidated=False)
ds_ww = xr.open_zarr(str(STORE), group=f"{SITE}/fluxnet_ww", consolidated=False)
ds_yy = xr.open_zarr(str(STORE), group=f"{SITE}/fluxnet_yy", consolidated=False)

for label, ds in [("root (HH)", ds_hh), ("fluxnet_dd", ds_dd),
                  ("fluxnet_mm", ds_mm), ("fluxnet_ww", ds_ww), ("fluxnet_yy", ds_yy)]:
    n_vars = len(ds.data_vars)
    dims   = dict(ds.dims)
    print(f"  {label:<14}  {n_vars:4d} vars   dims={dims}")

In [ ]:
# Inspect the 3-D NEE variable (time × ustar_threshold × nee_variant)
print(ds_hh["NEE"])
print()
print("ustar_threshold labels:", ds_hh["NEE"].ustar_threshold.values)
print("nee_variant labels    :", ds_hh["NEE"].nee_variant.values)

## 1 · Half-hourly meteorology (root group)

Air temperature and incoming shortwave radiation using xarray label-based indexing.

In [ ]:
site_id = ds_hh.attrs.get("site_id", SITE)

ta    = ds_hh["TA_F"].values.astype(float)     # Air temperature, gap-filled  [°C]
sw_in = ds_hh["SW_IN_F"].values.astype(float)  # Shortwave radiation in       [W m⁻²]
t_hh  = ds_hh["time"].values                   # numpy datetime64

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
fig.suptitle(f"{site_id} — Half-hourly meteorology (root group)", fontweight="bold")

axes[0].plot(t_hh, ta, lw=0.4, color="tab:red")
axes[0].set_ylabel("Air temperature (°C)")
axes[0].axhline(0, color="k", lw=0.5, ls="--")

axes[1].plot(t_hh, sw_in, lw=0.4, color="goldenrod")
axes[1].set_ylabel("SW$_{in}$ (W m⁻²)")

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    ax.grid(alpha=0.3)

fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## 2 · Daily carbon fluxes (`fluxnet_dd`)

**Top** — NEE with joint uncertainty envelope (VUT threshold, REF variant), using xarray `.sel()`.  
**Bottom** — GPP and RECO from both partitioning methods (NT = night-time, DT = day-time).

In [ ]:
t_dd = ds_dd["time"].values

# xarray label-based selection — no magic index numbers needed
nee     = ds_dd["NEE"].sel(ustar_threshold="VUT", nee_variant="REF").values.astype(float)
nee_unc = ds_dd["NEE_JOINTUNC"].sel(ustar_threshold="VUT", nee_variant="REF").values.astype(float)
gpp_nt  = ds_dd["GPP"].sel(ustar_threshold="VUT", partition_method="NT", nee_variant="REF").values.astype(float)
gpp_dt  = ds_dd["GPP"].sel(ustar_threshold="VUT", partition_method="DT", nee_variant="REF").values.astype(float)
reco_nt = ds_dd["RECO"].sel(ustar_threshold="VUT", partition_method="NT", nee_variant="REF").values.astype(float)
reco_dt = ds_dd["RECO"].sel(ustar_threshold="VUT", partition_method="DT", nee_variant="REF").values.astype(float)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
fig.suptitle(f"{site_id} — Daily carbon fluxes (fluxnet_dd, VUT threshold, REF variant)",
             fontweight="bold")

ax = axes[0]
ax.fill_between(t_dd, nee - nee_unc, nee + nee_unc, alpha=0.25, color="tab:blue", label="Joint unc.")
ax.plot(t_dd, nee, lw=1, color="tab:blue", label="NEE VUT REF")
ax.axhline(0, color="k", lw=0.6, ls="--")
ax.set_ylabel("NEE (µmol m⁻² s⁻¹)")
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(t_dd, gpp_nt,  lw=1, color="tab:green", label="GPP NT")
ax.plot(t_dd, gpp_dt,  lw=1, color="limegreen",  label="GPP DT", ls="--")
ax.plot(t_dd, reco_nt, lw=1, color="tab:brown",  label="RECO NT")
ax.plot(t_dd, reco_dt, lw=1, color="peru",        label="RECO DT", ls="--")
ax.set_ylabel("GPP / RECO (µmol m⁻² s⁻¹)")
ax.legend(fontsize=8, ncol=2)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    ax.grid(alpha=0.3)

fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## 3 · Monthly energy balance & soil profiles (`fluxnet_mm`)

**Top** — Latent heat (LE) and sensible heat (H) at the median energy-balance correction (p50).  
**Bottom** — Soil temperature and soil water content profiles across all measured layers.

In [ ]:
t_mm = ds_mm["time"].values

# Energy balance: pick p50 correction using label-based sel
le = ds_mm["LE_CORR"].sel(corr_pct="p50").values.astype(float)
h  = ds_mm["H_CORR"].sel(corr_pct="p50").values.astype(float)

# Soil profiles: TS and SWC have a soil_layer dimension
ts_all  = ds_mm["TS"].values.astype(float)   # shape (time, soil_layer)
swc_all = ds_mm["SWC"].values.astype(float)  # shape (time, soil_layer)
soil_labels = ds_mm["TS"].soil_layer.values

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
fig.suptitle(f"{site_id} — Monthly energy balance & soil profiles (fluxnet_mm)", fontweight="bold")

ax = axes[0]
ax.plot(t_mm, le, lw=1.5, color="tab:cyan",   label="LE (p50 EBC)")
ax.plot(t_mm, h,  lw=1.5, color="tab:orange", label="H  (p50 EBC)")
ax.axhline(0, color="k", lw=0.5, ls="--")
ax.set_ylabel("Energy flux (W m⁻²)")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

ax = axes[1]
n_ts  = ts_all.shape[1]
n_swc = swc_all.shape[1]
cmap_ts  = plt.cm.get_cmap("YlOrRd",  max(n_ts,  2))
cmap_swc = plt.cm.get_cmap("Blues",   max(n_swc, 3))
for i in range(n_ts):
    ax.plot(t_mm, ts_all[:, i],  lw=1, color=cmap_ts(i / max(n_ts - 1, 1)),
            label=f"TS {soil_labels[i]}")
for i in range(n_swc):
    ax.plot(t_mm, swc_all[:, i], lw=1, color=cmap_swc(0.4 + 0.6 * i / max(n_swc - 1, 1)),
            ls="--", label=f"SWC {soil_labels[i]}")
ax.set_ylabel("Soil T (°C) / SWC (%)")
ax.legend(fontsize=7, ncol=4)
ax.grid(alpha=0.3)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))

fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## 4 · Annual carbon budget (`fluxnet_yy`)

Bar chart of annual NEE, GPP, and RECO (VUT threshold, REF variant, NT partitioning).  
Negative NEE = net carbon uptake (sink); positive = source.

In [ ]:
t_yy  = ds_yy["time"].values
years = t_yy.astype("datetime64[Y]").astype(int) + 1970   # numpy datetime64 → year int

nee_y  = ds_yy["NEE"].sel(ustar_threshold="VUT", nee_variant="REF").values.astype(float)
gpp_y  = ds_yy["GPP"].sel(ustar_threshold="VUT", partition_method="NT", nee_variant="REF").values.astype(float)
reco_y = ds_yy["RECO"].sel(ustar_threshold="VUT", partition_method="NT", nee_variant="REF").values.astype(float)

x = np.arange(len(years))
w = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle(f"{site_id} — Annual carbon budget (fluxnet_yy, VUT REF, NT partitioning)",
             fontweight="bold")

ax.bar(x - w, nee_y,  w, label="NEE",  color="tab:blue",  alpha=0.85)
ax.bar(x,     gpp_y,  w, label="GPP",  color="tab:green", alpha=0.85)
ax.bar(x + w, reco_y, w, label="RECO", color="tab:brown", alpha=0.85)
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(years)
ax.set_ylabel("gC m⁻² yr⁻¹")
ax.legend()
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
plt.show()

## 5 · METEOSENS 4-D profile variables (root group)

Air temperature (`TA`) from the half-hourly root group is a 4-D variable with dimensions  
`(time, r, h, v)` — repeat/replicate, height, vertical position.  
Plot monthly means for each (r, h, v) combination that has valid data.

In [ ]:
ta4d = ds_hh["TA"]   # DataArray (time, TA_r, TA_h, TA_v)
print("TA shape :", ta4d.shape)
print("TA dims  :", ta4d.dims)

# Resample to monthly means, then plot one line per (r,h,v) combo with valid data
ta_monthly = ta4d.resample(time="ME").mean()

r_vals = ta4d.coords[ta4d.dims[1]].values
h_vals = ta4d.coords[ta4d.dims[2]].values
v_vals = ta4d.coords[ta4d.dims[3]].values
t_mo   = ta_monthly.time.values

fig, ax = plt.subplots(figsize=(14, 5))
fig.suptitle(f"{site_id} — Monthly mean air temperature by METEOSENS position (TA)",
             fontweight="bold")

n_plotted = 0
for ri, r in enumerate(r_vals):
    for hi, h in enumerate(h_vals):
        for vi, v in enumerate(v_vals):
            col = ta_monthly.isel(**{ta4d.dims[1]: ri,
                                     ta4d.dims[2]: hi,
                                     ta4d.dims[3]: vi}).values.astype(float)
            if np.all(np.isnan(col)):
                continue
            ax.plot(t_mo, col, lw=1, alpha=0.7, label=f"r={r} h={h} v={v}")
            n_plotted += 1

print(f"  Plotted {n_plotted} non-empty (r, h, v) combination(s)")
ax.set_ylabel("Air temperature (°C)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax.legend(fontsize=7, ncol=4, loc="upper right")
ax.grid(alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## 6 · Store summary — all variables and their shapes

In [ ]:
for label, ds in [("root (HH)", ds_hh), ("fluxnet_dd", ds_dd),
                  ("fluxnet_mm", ds_mm), ("fluxnet_ww", ds_ww), ("fluxnet_yy", ds_yy)]:
    nd = sum(1 for v in ds.data_vars if len(ds[v].dims) > 1)
    n1 = sum(1 for v in ds.data_vars if len(ds[v].dims) == 1)
    print(f"\n{'─'*60}")
    print(f"  {label}   ({nd} multi-dim, {n1} 1-D)")
    print(f"{'─'*60}")
    # multi-dim first, then 1-D, sorted
    for vname in sorted(ds.data_vars, key=lambda v: (-len(ds[v].dims), v)):
        da = ds[vname]
        units = da.attrs.get("units", "")
        print(f"  {vname:<30s}  {str(da.dims):<55s}  {str(da.shape):<25s}  {units}")